In [1]:
import requests
from IPython.display import display

base_url = "https://moustaphalabban.com/products.json"
all_products_data = []
page = 1
limit = 100

while True:
    params = {
        "page": page,
        "limit": limit
    }
    response = requests.get(base_url, params=params)
    response.raise_for_status()  # Raise an exception for HTTP errors
    data = response.json()

    products = data.get("products", [])

    if not products or len(products) < limit:
        # If no products or fewer than the limit, we've reached the last page
        for product in products:
            all_products_data.append({"id": product["id"], "handle": product["handle"]})
        break
    else:
        # Add all products from the current page
        for product in products:
            all_products_data.append({"id": product["id"], "handle": product["handle"]})

    page += 1

print(f"Found {len(all_products_data)} products.")
display(all_products_data)

Found 3824 products.


[{'id': 10162396299396, 'handle': 'pupa-wonder-me-shiny-bronzer'},
 {'id': 7203500097668, 'handle': 'pupa-wonder-me-blush'},
 {'id': 6590794039428, 'handle': 'pupa-wonder-cover-concealer'},
 {'id': 7203500064900, 'handle': 'pupa-vamp-stylo-liner'},
 {'id': 10162396168324, 'handle': 'pupa-vamp-eyeshadow-matt'},
 {'id': 10003214106756, 'handle': 'pupa-vamp-mascara-sexy-lashes'},
 {'id': 7203499901060,
  'handle': 'pupa-vamp-mascara-exceptional-volume-exagger'},
 {'id': 6583962730628, 'handle': 'pupa-vamp-mascara-all-in-one-101'},
 {'id': 10162397773956, 'handle': 'pupa-vamp-liquid-eyeshadow'},
 {'id': 8189990240388, 'handle': 'pupa-vamp-waterproof-2in1-lip-pencil'},
 {'id': 8189990109316, 'handle': 'pupa-vamp-waterproof-2in1-eye-pencil'},
 {'id': 10064654139524, 'handle': 'pupa-vamp-professional-liner-waterproof'},
 {'id': 7203499769988, 'handle': 'pupa-vamp-creamy-duo'},
 {'id': 7203499737220, 'handle': 'pupa-true-lips-pencil'},
 {'id': 10064654106756, 'handle': 'pupa-shock-plump-lip-gl

In [2]:
len(all_products_data)

3824

In [3]:
import requests
import json
import csv
from pathlib import Path
from typing import List, Dict

# ---------------- CONFIG ----------------
BASE_URL = "https://moustaphalabban.com/products/{}.js"
OUTPUT_JSON = "moustaphalabban_products.json"
OUTPUT_CSV = "moustaphalabban_products.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}


# ---------------- FETCH FUNCTION ----------------
def fetch_product(handle: str) -> Dict:
    url = BASE_URL.format(handle)
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"[ERROR] Failed to fetch {handle}: {e}")
        return None


# ---------------- PARSE FUNCTION ----------------
def parse_product(product: Dict) -> List[Dict]:
    results = []

    # Product-level fields
    product_data = {
        "product_id": product.get("id"),
        "title": product.get("title"),
        "handle": product.get("handle"),
        "description": product.get("description"),
        "created_at": product.get("created_at"),
        "vendor": product.get("vendor"),
        "type": product.get("type"),
        "price": product.get("price"),
        "compare_at_price": product.get("compare_at_price"),
        "available": product.get("available"),
    }

    variants = product.get("variants", [])

    # If NO variants → still return one row
    if not variants:
        results.append(product_data)
        return results

    # Variant-level fields
    for variant in variants:
        row = product_data.copy()

        row.update({
            "variant_id": variant.get("id"),
            "variant_title": variant.get("title"),
            "variant_sku": variant.get("sku"),
            "variant_available": variant.get("available"),
            "variant_name": variant.get("name"),
            "variant_price": variant.get("price"),
            "variant_compare_at_price": variant.get("compare_at_price"),
            "variant_barcode": variant.get("barcode"),
            "variant_image": variant.get("featured_image", {}).get("src") if variant.get("featured_image") else None
        })

        results.append(row)

    return results


# ---------------- MAIN ----------------
def main():
    all_data = []

    for item in all_products_data:
        handle = item["handle"]
        print(f"[INFO] Fetching {handle}")

        product_json = fetch_product(handle)
        if not product_json:
            continue

        parsed_rows = parse_product(product_json)
        all_data.extend(parsed_rows)

    # Save JSON
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(all_data, f, ensure_ascii=False, indent=2)

    # Save CSV
    if all_data:
        keys = all_data[0].keys()
        with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            writer.writerows(all_data)

    print(f"[DONE] Saved {len(all_data)} rows")


# ---------------- RUN ----------------
if __name__ == "__main__":
    main()

[INFO] Fetching pupa-wonder-me-shiny-bronzer
[INFO] Fetching pupa-wonder-me-blush
[INFO] Fetching pupa-wonder-cover-concealer
[INFO] Fetching pupa-vamp-stylo-liner
[INFO] Fetching pupa-vamp-eyeshadow-matt
[INFO] Fetching pupa-vamp-mascara-sexy-lashes
[INFO] Fetching pupa-vamp-mascara-exceptional-volume-exagger
[INFO] Fetching pupa-vamp-mascara-all-in-one-101
[INFO] Fetching pupa-vamp-liquid-eyeshadow
[INFO] Fetching pupa-vamp-waterproof-2in1-lip-pencil
[INFO] Fetching pupa-vamp-waterproof-2in1-eye-pencil
[INFO] Fetching pupa-vamp-professional-liner-waterproof
[INFO] Fetching pupa-vamp-creamy-duo
[INFO] Fetching pupa-true-lips-pencil
[INFO] Fetching pupa-shock-plump-lip-gloss
[INFO] Fetching pupa-shine-bright-all-in-one-palette
[INFO] Fetching pupa-fruit-lovers-body-scrub
[INFO] Fetching pupa-professionals-bb-cream-primer-nude
[INFO] Fetching pupa-professionals-bb-cream-primer-all-skin-types
[INFO] Fetching pupa-plump-care-eyebrow-gel
[INFO] Fetching pupa-pleasure-lip-oil
[INFO] Fetchin

In [6]:
import requests
from bs4 import BeautifulSoup
import json
import csv

# ---------------- CONFIG ----------------
BASE_URL = "https://www.fahsbeauty.com/shop/page/{}/"
START_PAGE = 1   # you can change this
END_PAGE = 80     # increase if you want multiple pages

OUTPUT_JSON = "fahs_products.json"
OUTPUT_CSV = "fahs_products.csv"

HEADERS = {
    "accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "accept-encoding": "gzip, deflate, br, zstd",
    "accept-language": "en-US,en;q=0.9,id;q=0.8,fa;q=0.7,ar;q=0.6,ms;q=0.5,ja;q=0.4,es;q=0.3",
    "cookie": "tk_ai=o0u360Q2F4oWFgf4L8YdAMaA",
    "priority": "u=0, i",
    "referer": "https://www.fahsbeauty.com/shop/",
    "sec-ch-ua": '"Chromium";v="146", "Not-A.Brand";v="24", "Microsoft Edge";v="146"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"Windows"',
    "sec-fetch-dest": "document",
    "sec-fetch-mode": "navigate",
    "sec-fetch-site": "same-origin",
    "sec-fetch-user": "?1",
    "upgrade-insecure-requests": "1",
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36 Edg/146.0.0.0"
}


# ---------------- FETCH ----------------
def fetch_page(page):
    url = BASE_URL.format(page)
    try:
        res = requests.get(url, headers=HEADERS, timeout=10)
        res.raise_for_status()
        return res.text
    except Exception as e:
        print(f"[ERROR] Page {page}: {e}")
        return None


# ---------------- PARSE ----------------
def parse_products(html):
    soup = BeautifulSoup(html, "html.parser")
    results = []

    products = soup.select("div.c-product-grid__item")

    for product in products:
        # product title
        title_tag = product.select_one("h2.woocommerce-loop-product__title")

        # product link
        link_tag = product.select_one("a.woocommerce-LoopProduct-link")

        if not title_tag or not link_tag:
            continue

        name = title_tag.get_text(strip=True)
        url = link_tag.get("href")

        # Fix relative URLs
        if url.startswith("/"):
            url = "https://www.fahsbeauty.com" + url

        results.append({
            "product_name": name,
            "product_url": url
        })

    return results


# ---------------- MAIN ----------------
def main():
    all_products = []

    for page in range(START_PAGE, END_PAGE + 1):
        print(f"[INFO] Scraping page {page}")
        html = fetch_page(page)

        if not html:
            continue

        products = parse_products(html)
        all_products.extend(products)

    # Save JSON
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(all_products, f, indent=2, ensure_ascii=False)

    # Save CSV
    if all_products:
        keys = all_products[0].keys()
        with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            writer.writerows(all_products)

    print(f"[DONE] Total products: {len(all_products)}")


# ---------------- RUN ----------------
if __name__ == "__main__":
    main()

[INFO] Scraping page 1
[ERROR] Page 1: 403 Client Error: Forbidden for url: https://www.fahsbeauty.com/shop/page/1/
[INFO] Scraping page 2
[ERROR] Page 2: 403 Client Error: Forbidden for url: https://www.fahsbeauty.com/shop/page/2/
[INFO] Scraping page 3
[ERROR] Page 3: 403 Client Error: Forbidden for url: https://www.fahsbeauty.com/shop/page/3/
[INFO] Scraping page 4
[ERROR] Page 4: 403 Client Error: Forbidden for url: https://www.fahsbeauty.com/shop/page/4/
[INFO] Scraping page 5
[ERROR] Page 5: 403 Client Error: Forbidden for url: https://www.fahsbeauty.com/shop/page/5/
[INFO] Scraping page 6
[ERROR] Page 6: 403 Client Error: Forbidden for url: https://www.fahsbeauty.com/shop/page/6/
[INFO] Scraping page 7
[ERROR] Page 7: 403 Client Error: Forbidden for url: https://www.fahsbeauty.com/shop/page/7/
[INFO] Scraping page 8
[ERROR] Page 8: 403 Client Error: Forbidden for url: https://www.fahsbeauty.com/shop/page/8/
[INFO] Scraping page 9
[ERROR] Page 9: 403 Client Error: Forbidden for u

In [ ]:
import json
import csv
from pathlib import Path
from bs4 import BeautifulSoup

# ---------------- CONFIG ----------------
INPUT_DIR = Path("product_pages/product_pages")
OUTPUT_JSON = "parsed_products.json"
OUTPUT_CSV = "parsed_products.csv"

BASE_URL = "https://www.fahsbeauty.com/shop/"


# ---------------- HELPERS ----------------
def get_text(el):
    return el.get_text(strip=True) if el else None


def get_all_text(el):
    return el.get_text(separator=" ", strip=True) if el else None


def extract_product(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    data = {}

    # ---------------- BASIC ----------------
    data["product_url"] = BASE_URL + file_path.stem + "/"

    # Title
    data["title"] = get_text(soup.select_one("h1.c-product__title"))

    # Short description
    data["short_description"] = get_all_text(
        soup.select_one("div.c-product__short-description")
    )

    # Full description
    data["description"] = get_all_text(
        soup.select_one("#tab-description")
    )

    # ---------------- BRAND ----------------
    brand = soup.select_one(".c-product__brands a")
    if not brand:
        brand = soup.select_one(".c-product__brand-logo-list a img")
        data["brand"] = brand.get("alt") if brand else None
    else:
        data["brand"] = get_text(brand)

    # ---------------- SKU ----------------
    data["sku"] = get_text(soup.select_one(".sku"))

    # ---------------- TAGS ----------------
    tags = soup.select(".tagged_as a")
    data["tags"] = [get_text(t) for t in tags] if tags else []

    # ---------------- CATEGORIES ----------------
    cats = soup.select(".posted_in a")
    data["categories"] = [get_text(c) for c in cats] if cats else []

    # ---------------- PRICE ----------------
    price = soup.select_one(".price .woocommerce-Price-amount")
    data["price"] = get_text(price)

    # ---------------- IMAGE ----------------
    img = soup.select_one(".c-product__slider-item a")
    data["image_url"] = img.get("href") if img else None

    return data


# ---------------- MAIN ----------------
def main():
    all_products = []

    files = list(INPUT_DIR.glob("*.html"))

    print(f"[INFO] Found {len(files)} files")

    for i, file in enumerate(files, 1):
        try:
            print(f"[{i}/{len(files)}] Parsing {file.name}")
            product = extract_product(file)
            all_products.append(product)

        except Exception as e:
            print(f"[ERROR] {file.name}: {e}")

    # Save JSON
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(all_products, f, indent=2, ensure_ascii=False)

    # Save CSV
    if all_products:
        keys = all_products[0].keys()
        with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            writer.writerows(all_products)

    print(f"\n[DONE] Parsed {len(all_products)} products")


# ---------------- RUN ----------------
if __name__ == "__main__":
    main()

[INFO] Found 2394 files
[1/2394] Parsing initio-musk-theraph-extrait-de-parfum.html
[2/2394] Parsing shi-refreshing-cleansing-sheets.html
[3/2394] Parsing carolina-herrera-good-girl-blush-elixir-women-eau-de-parfum.html
[4/2394] Parsing estee-lauder-pure-color-envy-liquid-lipcolor-matte-304-quiet-riot-7ml.html
[5/2394] Parsing jaguar-classic-motion-man-eau-de-toilette-100ml.html
[6/2394] Parsing jaguar-classic-gold-man-eau-de-toilette-100ml.html
[7/2394] Parsing max-factor-masterpiece-matte-liquid-eyeliner-espresso-03.html
[8/2394] Parsing set-afnan-9pm-man.html
[9/2394] Parsing elie-saab-le-parfum-lumiere-women-vapo-90ml.html
[10/2394] Parsing clarins-milky-boost-foundation-05-50ml.html
[11/2394] Parsing dr-vranjes-firenze-oud-nobile-reed-diffuser-500ml.html
[12/2394] Parsing tom-ford-noir-de-noir-prive-eau-de-parfum.html
[13/2394] Parsing caron-pour-un-homme-de-caron-eau-de-toilette-750ml.html
[14/2394] Parsing set-lancome-ladies-idole-gift-set-fragrances.html
[15/2394] Parsing guerl

In [1]:
!unzip -o /content/product_pages.zip -d /content/product_pages

Archive:  /content/product_pages.zip
   creating: /content/product_pages/product_pages/
  inflating: /content/product_pages/product_pages/4711-man-eau-de-toilette-400ml.html  
  inflating: /content/product_pages/product_pages/afnan-9-am-100ml-edp.html  
  inflating: /content/product_pages/product_pages/afnan-9-am-dive-homme-edp-100ml.html  
  inflating: /content/product_pages/product_pages/afnan-9-am-femme-edp-100ml.html  
  inflating: /content/product_pages/product_pages/afnan-9-pm-100ml-edp.html  
  inflating: /content/product_pages/product_pages/afnan-9-pm-elixir.html  
  inflating: /content/product_pages/product_pages/afnan-9-pm-femme-edp-100ml.html  
  inflating: /content/product_pages/product_pages/afnan-9pm-rebel-man-100ml.html  
  inflating: /content/product_pages/product_pages/afnan-cherry-bouquet-women-eau-de-perfume-80-ml.html  
  inflating: /content/product_pages/product_pages/afnan-concentrated-perfume-musk-abiyad-oil-20ml.html  
  inflating: /content/product_pages/product